In [ ]:
import pandas as pd
import re
from sklearn.metrics import cohen_kappa_score

# ---------- CONFIG ----------
RATER1_FILE = "rater1.csv"
RATER2_FILE = "rater2.csv"

RATER1_CLEAN_OUT = "rater1_clean.csv"
RATER2_CLEAN_OUT = "rater2_clean.csv"

# ---------- HELPERS ----------
def extract_score(cell):
    """Extract leading numeric score from '2=Mostly Informative'"""
    if pd.isna(cell):
        return None
    match = re.match(r'(\d+)', str(cell))
    return int(match.group(1)) if match else None


def process_file(file_path):
    df = pd.read_csv(file_path, header=None)

    records = []
    current_model = None
    current_setting = None
    current_criterion = None

    for _, row in df.iterrows():
        model = row[0]
        setting = row[1]
        criterion = row[3]
        item_id = row[4]

        if pd.notna(model):
            current_model = model
        if pd.notna(setting):
            current_setting = setting
        if pd.notna(criterion):
            current_criterion = criterion

        if pd.notna(item_id):
            scores = [extract_score(row[i]) for i in range(6, 9)]

            records.append({
                "model": current_model,
                "setting": current_setting,
                "criterion": current_criterion,
                "item_id": int(item_id),
                "info": scores[0],
                "coherence": scores[1],
                "usefulness": scores[2]
            })

    return pd.DataFrame(records)


# ---------- PROCESS + SAVE CLEAN FILES ----------
r1 = process_file(RATER1_FILE)
r2 = process_file(RATER2_FILE)

r1.to_csv(RATER1_CLEAN_OUT, index=False)
r2.to_csv(RATER2_CLEAN_OUT, index=False)

print(f"Saved cleaned rater1 → {RATER1_CLEAN_OUT}")
print(f"Saved cleaned rater2 → {RATER2_CLEAN_OUT}")

# ---------- RENAME FOR MERGE ----------
r1 = r1.rename(columns={
    "info": "r1_info",
    "coherence": "r1_coherence",
    "usefulness": "r1_usefulness"
})

r2 = r2.rename(columns={
    "info": "r2_info",
    "coherence": "r2_coherence",
    "usefulness": "r2_usefulness"
})

# ---------- MERGE ----------
merged = pd.merge(
    r1, r2,
    on=["model", "setting", "criterion", "item_id"]
)

# ---------- FLATTEN ----------
r1_all = pd.concat([
    merged["r1_info"],
    merged["r1_coherence"],
    merged["r1_usefulness"]
], ignore_index=True)

r2_all = pd.concat([
    merged["r2_info"],
    merged["r2_coherence"],
    merged["r2_usefulness"]
], ignore_index=True)

# ---------- KAPPA ----------
kappa = cohen_kappa_score(r1_all, r2_all, weights="quadratic")

print("\n============================")
print(f"Overall Cohen's Kappa: {kappa:.4f}")
print("============================")